In [ ]:
import os
import json
import pandas as pd

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv("../src/.env")

with open("../config/config.json", "r") as f:
    config = json.load(f)

df = pd.read_csv("../data/atc_pilot_transcripts.csv")

print("Data Loaded")
print(df.head(), "\n")

print("LLM_PROVIDER from env:", os.getenv("LLM_PROVIDER"))

def load_llm(config):
   # provider = os.getenv("LLM_PROVIDER")
    provider = config["LLM_PROVIDER"]

    if provider == "openai":
        llm = ChatOpenAI(
            model=config["llm"]["openai_model"],
            temperature=config["llm"]["temperature"]
        )

    elif provider == "gemini":
        llm = ChatGoogleGenerativeAI(
            model=config["llm"]["gemini_model"],
            temperature=config["llm"]["temperature"]
        )

    else:
        raise ValueError("Invalid LLM_PROVIDER")

    return llm
llm = load_llm(config)
print(f" LLM Initialized using provider: {os.getenv('LLM_PROVIDER')}")
print(f" LLM Initialized using provider: {config['LLM_PROVIDER']}")

# --- Step 6: Test LLM ---
response = llm.invoke("Say 'setup successful' in one short sentence.")
print("\n LLM Response:")
print(response.content)




In [ ]:
# --- Step 1: Imports ---
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


# --- Step 2: Define Output Schema ---

class ClassificationOutput(BaseModel):

    call_type: str = Field(
        description=(
            "Classification of the cockpit/ATC transcript. "
            "Must be one of the configured classification labels."
        )
    )

    confidence: float = Field(
        description=(
            "Confidence score between 0 and 1 indicating "
            "confidence in the classification."
        )
    )


# Create Pydantic parser
parser = PydanticOutputParser(
    pydantic_object=ClassificationOutput
)


# --- Step 3: Prompt Template ---

prompt = PromptTemplate(
    template="""
You are an AI-powered QA classification assistant
for an Airline Cockpit Voice Recorder / ATC communication
evaluation system.

Your task is to classify the following cockpit/ATC transcript
into exactly ONE of the following categories:

{labels}

Classification definitions:

- routine_operations:
  Normal flight or ATC communication without a significant
  abnormal condition.

- minor_abnormal_event:
  A non-critical abnormal situation, deviation, or issue
  requiring attention but not representing a critical emergency.

- critical_emergency:
  A serious or potentially life-threatening emergency
  requiring immediate attention.

- weather_deviation:
  Communication primarily related to weather conditions,
  weather avoidance, or deviation due to weather.

IMPORTANT RULES:

1. Select exactly ONE classification label.
2. Use only the information available in the transcript.
3. Do not invent events or information.
4. Return a confidence score between 0 and 1.
5. If the transcript is ambiguous, select the closest
   category and reduce the confidence score.
6. Return ONLY the structured output requested below.

Transcript:
-------------------------
{transcript}
-------------------------

{format_instructions}
""",

    input_variables=["transcript"],

    partial_variables={
        "format_instructions": parser.get_format_instructions(),
        "labels": ", ".join(
            config["classification"]["labels"]
        )
    }
)


# --- Step 4: Create Chain ---

classification_chain = prompt | llm | parser


# --- Step 5: Test with One Transcript ---

sample_text = df.iloc[0]["transcript"]

result = classification_chain.invoke({
    "transcript": sample_text
})


# --- Step 6: Display Result ---

print("\n" + "=" * 60)
print("✈ AIRLINE COCKPIT QA - CLASSIFICATION RESULT")
print("=" * 60)

print(f"Call Type   : {result.call_type}")
print(f"Confidence  : {result.confidence:.2f}")

print("=" * 60)

In [ ]:
from tqdm import tqdm
import pandas as pd

results = []

for i, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Classifying Cockpit Transcripts"
):
    try:
        # Send transcript to LLM classification chain
        output = classification_chain.invoke({
            "transcript": row["transcript"]
        })

        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": output.call_type,
            "confidence": output.confidence
        })

    except Exception as e:
        print(f" Error at row {i}: {e}")

        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": None,
            "confidence": None
        })


# Convert results to DataFrame
results_df = pd.DataFrame(results)


# Merge classification results with original dataset
df = df.merge(
    results_df,
    on="call_id"
)


print("\n Batch Cockpit Classification Completed\n")

print(
    df[
        [
            "call_id",
            "expected_call_type",
            "predicted_call_type",
            "confidence"
        ]
    ]
)

In [ ]:
# --- Step 1: Define routing logic ---

def route_call(call_type):

    if call_type == "routine_operations":
        return [
            "readback_hearback_accuracy",
            "phraseology_compliance",
            "crm_coordination",
            "situational_awareness"
        ]

    elif call_type == "minor_abnormal_event":
        return [
            "readback_hearback_accuracy",
            "phraseology_compliance",
            "crm_coordination",
            "situational_awareness"
        ]

    elif call_type == "critical_emergency":
        return [
            "readback_hearback_accuracy",
            "crm_coordination",
            "situational_awareness"
        ]

    elif call_type == "weather_deviation":
        return [
            "readback_hearback_accuracy",
            "phraseology_compliance",
            "situational_awareness"
        ]

    else:
        # Fallback evaluation criteria
        return [
            "readback_hearback_accuracy",
            "phraseology_compliance"
        ]


# --- Step 2: Apply routing to dataset ---

df["evaluation_plan"] = (
    df["predicted_call_type"]
    .apply(route_call)
)


# --- Step 3: Display routing results ---

print("✅ Airline QA Evaluation Routing Applied\n")

print(
    df[
        [
            "call_id",
            "predicted_call_type",
            "evaluation_plan"
        ]
    ]
)

In [ ]:
# --- Step 1: Imports ---

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


# --- Step 2: Define Output Schema ---

class QAEvaluation(BaseModel):

    score: int = Field(
        description="QA evaluation score between 1 and 5"
    )

    reasoning: str = Field(
        description="Detailed explanation supporting the assigned score"
    )

    evidence: str = Field(
        description=(
            "Specific evidence from the transcript that supports "
            "the evaluation"
        )
    )


# Create Pydantic parser
qa_parser = PydanticOutputParser(
    pydantic_object=QAEvaluation
)


# --- Step 3: Prompt Template ---

qa_prompt = PromptTemplate(
    template="""
You are a QA evaluator for an Airline Cockpit Voice
Recorder / ATC-Pilot communication evaluation system.

Evaluate the following cockpit/ATC transcript based on
the specified QA criterion.

QA Criterion:
{criterion}

Evaluation Criteria:

1. readback_hearback_accuracy
   Evaluate whether instructions, clearances, headings,
   altitudes, frequencies, or other relevant information
   were correctly heard and read back.

2. phraseology_compliance
   Evaluate whether the communication follows the expected
   aviation communication phraseology and terminology
   represented in the transcript.

3. crm_coordination
   Evaluate the effectiveness of coordination between
   cockpit crew members and other relevant participants
   represented in the transcript.

4. situational_awareness
   Evaluate whether the participants demonstrate awareness
   of the relevant flight/communication situation based
   on the information available in the transcript.

Scoring:

1 = Very Poor
2 = Poor
3 = Average / Partially Meets Expectations
4 = Good
5 = Excellent

IMPORTANT:

- Evaluate ONLY the selected QA criterion.
- Use only information available in the transcript.
- Do not invent missing information.
- Do not assume that an action occurred if it is not
  supported by the transcript.
- Give a score from 1 to 5.
- Provide specific reasoning.
- Quote or reference relevant transcript evidence.
- If there is insufficient evidence, clearly mention this
  in the reasoning.

Transcript:
-------------------------
{transcript}
-------------------------

Return the result using the following structured format:

{format_instructions}
""",

    input_variables=[
        "transcript",
        "criterion"
    ],

    partial_variables={
        "format_instructions": qa_parser.get_format_instructions()
    }
)


# --- Step 4: Create Evaluation Chain ---

qa_chain = qa_prompt | llm | qa_parser

evaluation_plan = df.iloc[0]["evaluation_plan"]

sample_text = df.iloc[0]["transcript"]

evaluation_results = []

for criterion in evaluation_plan:

    try:

        result = qa_chain.invoke({
            "transcript": sample_text,
            "criterion": criterion
        })

        evaluation_results.append({
            "criterion": criterion,
            "score": result.score,
            "reasoning": result.reasoning,
            "evidence": result.evidence
        })

    except Exception as e:

        print(
            f"❌ Error evaluating "
            f"{criterion}: {e}"
        )


print("\n✈️ Evaluation Results")

for result in evaluation_results:

    print(
        f"\n{result['criterion']}"
    )

    print(
        f"Score: {result['score']}/5"
    )

    print(
        f"Reasoning: {result['reasoning']}"
    )

    print(
        f"Evidence: {result['evidence']}"
    )

In [ ]:
# --- Step 1: Schema ---
class AirlineQAEvaluation(BaseModel):
    score: int = Field(
        description="QA evaluation score between 1 and 5"
    )
    reasoning: str = Field(
        description="Explanation of the score based on the transcript"
    )

airline_qa_parser = PydanticOutputParser(
    pydantic_object=AirlineQAEvaluation
)


# --- Step 2: Prompt ---
airline_qa_prompt = PromptTemplate(
    template="""
You are a QA evaluator for an Airline Cockpit Voice Recorder
and ATC-Pilot communication evaluation system.

Evaluate the transcript based on the following QA criterion:

QA Criterion:
{criterion}

Consider the following when evaluating:

- readback_hearback_accuracy:
  Did the pilot correctly hear and read back ATC instructions,
  clearances, headings, altitudes, frequencies, or other
  relevant information?

- phraseology_compliance:
  Did the pilot/ATC communication follow the expected
  aviation phraseology and terminology?

- crm_coordination:
  Was there effective coordination between cockpit crew
  members and other relevant participants?

- situational_awareness:
  Did the participants demonstrate awareness of the
  current flight or communication situation?

Scoring:

1 = Very Poor
2 = Poor
3 = Average / Partially Meets Expectations
4 = Good
5 = Excellent

Important instructions:

- Evaluate ONLY the selected QA criterion.
- Use only information available in the transcript.
- Do not invent information or actions.
- Provide a score between 1 and 5.
- Explain the reason for the assigned score.
- If there is insufficient evidence, clearly mention it.

Transcript:
-------------------------
{transcript}
-------------------------

{format_instructions}
""",
    input_variables=["transcript", "criterion"],
    partial_variables={
        "format_instructions":
            airline_qa_parser.get_format_instructions()
    }
)


# --- Step 3: Chain ---
airline_qa_chain = (
    airline_qa_prompt
    | llm
    | airline_qa_parser
)


# --- Step 4: Test ---
sample_text = df.iloc[0]["transcript"]

result = airline_qa_chain.invoke({
    "transcript": sample_text,
    "criterion": "readback_hearback_accuracy"
})

print("✅ Airline QA Evaluation Result:")
print(result)

In [ ]:
from tqdm import tqdm
import pandas as pd
import pprint


# --- Step 1: Evaluation Runner ---

def run_evaluations(transcript, eval_plan):
    results = {}

    for criterion in eval_plan:
        try:
            result = airline_qa_chain.invoke({
                "transcript": transcript,
                "criterion": criterion
            })

            results[criterion] = result.model_dump()

        except Exception as e:
            results[criterion] = {
                "error": str(e)
            }

    return results


# --- Step 2: Apply Evaluations to Entire Dataset ---

evaluation_outputs = []

for i, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Running Airline QA Evaluations"
):

    output = run_evaluations(
        row["transcript"],
        row["evaluation_plan"]
    )

    evaluation_outputs.append({
        "call_id": row["call_id"],
        "evaluation_output": output
    })


# --- Step 3: Convert to DataFrame ---

eval_df = pd.DataFrame(evaluation_outputs)


# --- Step 4: Merge Evaluation Results ---

df = df.merge(
    eval_df,
    on="call_id"
)


print("\n✅ Airline QA Evaluation Completed\n")


# --- Step 5: Show One Example ---

pprint.pprint(
    df.iloc[0]["evaluation_output"]
)

In [ ]:
# --- Step 1: Schema ---

class FinalAirlineQAReport(BaseModel):
    summary: str = Field(
        description="Overall summary of the pilot/ATC communication QA performance"
    )
    recommendations: list[str] = Field(
        description="List of specific and actionable improvements"
    )


final_parser = PydanticOutputParser(
    pydantic_object=FinalAirlineQAReport
)


# --- Step 2: Prompt ---

final_prompt = PromptTemplate(
    template="""
You are a QA manager reviewing an Airline Cockpit Voice Recorder
and ATC-Pilot communication transcript.

Based on the evaluation results below, generate:

1. A concise overall summary of the communication performance.
2. A list of specific and actionable recommendations for improvement.

The evaluation may contain the following QA criteria:

- readback_hearback_accuracy
- phraseology_compliance
- crm_coordination
- situational_awareness

Evaluation Data:
-------------------------
{evaluation_output}
-------------------------

IMPORTANT:

- Base the report ONLY on the evaluation data provided.
- Do not invent events, actions, or problems.
- Identify the main strengths and weaknesses.
- Recommendations should be specific and practical.
- Focus recommendations on areas that need improvement.
- Do not simply repeat the scores.
- If there is insufficient evidence for a recommendation,
  do not invent one.
- Keep the summary concise and professional.

{format_instructions}
""",
    input_variables=["evaluation_output"],
    partial_variables={
        "format_instructions": final_parser.get_format_instructions()
    }
)


# --- Step 3: Chain ---

final_chain = final_prompt | llm | final_parser


# --- Step 4: Test on One Row ---

sample_eval = df.iloc[0]["evaluation_output"]

result = final_chain.invoke({
    "evaluation_output": sample_eval
})


print("\n" + "=" * 60)
print("✈️ FINAL AIRLINE QA REPORT")
print("=" * 60)

print("\nSummary:")
print(result.summary)

print("\nRecommendations:")
for recommendation in result.recommendations:
    print(f"- {recommendation}")

print("=" * 60)

In [17]:
from tqdm import tqdm
import pandas as pd
from pathlib import Path

# --- Step 1: Generate Final Airline QA Reports ---

final_outputs = []

for i, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Generating Final Airline QA Reports"
):
    try:
        result = final_chain.invoke({
            "evaluation_output": row["evaluation_output"]
        })

        final_outputs.append({
            "call_id": row["call_id"],
            "summary": result.summary,
            "recommendations": result.recommendations
        })

    except Exception as e:
        print(f"❌ Error at row {i}: {e}")

        final_outputs.append({
            "call_id": row["call_id"],
            "summary": None,
            "recommendations": None
        })


# --- Step 2: Convert to DataFrame ---

final_df = pd.DataFrame(final_outputs)


# --- Step 3: Remove old generated columns ---

df = df.drop(
    columns=["summary", "recommendations"],
    errors="ignore"
)


# --- Step 4: Merge Final Reports with Original DataFrame ---

df = df.merge(
    final_df,
    on="call_id",
    how="left"
)


print("\n✅ Final Airline QA Reports Generated\n")


# --- Step 5: Display Final Results ---

print(
    df[[
        "call_id",
        "predicted_call_type",
        "evaluation_output",
        "summary",
        "recommendations"
    ]]
)


# --- Step 6: Save to Excel ---

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "output.xlsx"

df.to_excel(output_file, index=False)

print(f"\n✅ Results saved successfully to: {output_file}")

Generating Final Airline QA Reports: 100%|██████████| 30/30 [01:30<00:00,  3.02s/it]


✅ Final Airline QA Reports Generated

    call_id   predicted_call_type  \
0         1    routine_operations   
1         2    routine_operations   
2         3    routine_operations   
3         4    routine_operations   
4         5    routine_operations   
5         6    routine_operations   
6         7    routine_operations   
7         8  minor_abnormal_event   
8         9     weather_deviation   
9        10  minor_abnormal_event   
10       11    routine_operations   
11       12  minor_abnormal_event   
12       13  minor_abnormal_event   
13       14    critical_emergency   
14       15    routine_operations   
15       16     weather_deviation   
16       17     weather_deviation   
17       18    routine_operations   
18       19    routine_operations   
19       20    routine_operations   
20       21    routine_operations   
21       22    critical_emergency   
22       23    routine_operations   
23       24    routine_operations   
24       25  minor_abnormal_event   